# 已失败分钟回撤版：保存模型预测到订单的只读链路

冻结比较全部2021/2022八季度，以2022Q2为弱区间、2021Q3为相对较强区间。只复用Qlib Recorder的模型、预测、日动作及固定股数五会话标签；每季报告首个09:48分钟的模型分数、实际费后值、费用和候选/控制动作。系数×标准化输入均值变化只解释保存模型算术，不选交易阈值。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
warnings.filterwarnings('ignore',message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.analysis.stage1_fixed_share_drawdown_information import prepare_feature_panel
root=Path.cwd();card=root/'research/stage1_v5_drawdown_failure_trace_20261030';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['provider_uri']/'qlib_store_manifest.json')==plan['provider_manifest_sha256'] and sha(root/plan['feature_code'])==plan['feature_code_sha256']
info=root/plan['information_card'];assert sha(info/'summary.json')==plan['information_summary_sha256'] and sha(info/'verification_manifest.json')==plan['information_manifest_sha256'] and json.loads((info/'nas_binding.json').read_text())['evidence']['sha256']==plan['information_nas_archive_sha256']
for role in ('label_source','parent','frame'):
 s=plan[role];source=root/s['card'];assert sha(source/'summary.json')==s['summary_sha256'] and sha(source/'verification_manifest.json')==s['manifest_sha256'] and json.loads((source/'nas_binding.json').read_text())['evidence']['sha256']==s['nas_archive_sha256']
qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
def read(spec,name):
 s=spec;v,_=load_verified_recorder_objects(root/s['card'],expected_recorder_id=s['recorder_id'],artifacts=[name],expected_manifest_sha256=s['manifest_sha256']);return v[name]
info_spec={'card':plan['information_card'],'recorder_id':plan['information_recorder_id'],'manifest_sha256':plan['information_manifest_sha256']}
models={name:read(info_spec,'models/'+name) for name in ('control_soxx','candidate_soxx')}
pred=read(info_spec,'analysis/predictions');daily=read(info_spec,'analysis/daily_actions');labels=read(plan['label_source'],'analysis/labels');book=read(plan['parent'],'analysis/native/dated_f696_0');frame=read(plan['frame'],'inputs/covered_frame')
source_plan=json.loads((info/'plan.json').read_text())
assert len(pred)==180938 and len(daily)==494
print('verified Qlib/NAS model and action sources')

[40306:MainThread](2026-10-30 15:47:07,042) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[40306:MainThread](2026-10-30 15:47:07,493) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[40306:MainThread](2026-10-30 15:47:07,493) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


verified Qlib/NAS model and action sources


In [2]:
bars=D.features(['SOXL','SOXX'],['$close','$raw_volume','$price_valid'],start_time='2018-10-15 09:31:00',end_time='2022-12-22 16:00:00',freq='1min')
panel,source=prepare_feature_panel(labels,frame,book,bars,source_plan)
eval_panel=panel.loc[panel.trade_date.dt.year.isin([2021,2022]) & panel.trade_date.le(pd.Timestamp('2022-12-16'))].reset_index(drop=True)
assert len(eval_panel)==len(pred)==180938 and eval_panel.trade_date.reset_index(drop=True).equals(pd.to_datetime(pred.trade_date).reset_index(drop=True))
labels_eval=labels.loc[labels.trade_date.dt.year.isin([2021,2022]) & labels.trade_date.le(pd.Timestamp('2022-12-16'))].reset_index(drop=True)
assert eval_panel[['trade_date','bar_end_at']].equals(labels_eval[['trade_date','bar_end_at']]) and labels_eval.status.eq('paired_funded').all()
features=[*source_plan['base39_features'],*source_plan['state_features'],*source_plan['ordinary_minute_features']]
candidate=[*features,source_plan['candidate_only_feature']]
xc=pd.concat({'feature':eval_panel[candidate]},axis=1);xb=pd.concat({'feature':eval_panel[features]},axis=1)
model=models['candidate_soxx'];control=models['control_soxx']
calc=model.predict(xc);base=control.predict(xb)
max_prediction_error=max(float(np.max(np.abs(calc-pred.candidate_soxx.to_numpy(float)))),float(np.max(np.abs(base-pred.control_soxx.to_numpy(float)))))
assert max_prediction_error<1e-12
z=model.named_steps['standardscaler'].transform(model.named_steps['simpleimputer'].transform(xc))
coef=model.named_steps['ridge'].coef_;intercept=float(model.named_steps['ridge'].intercept_)
manual=intercept+z@coef;max_manual_error=float(np.max(np.abs(manual-calc)));assert max_manual_error<1e-12
names=list(model.named_steps['simpleimputer'].get_feature_names_out(candidate));assert len(names)==len(coef)
first=(eval_panel.bar_end_at.dt.hour.eq(9)&eval_panel.bar_end_at.dt.minute.eq(48)).to_numpy();assert first.sum()==494
firstbar=eval_panel.loc[first,['trade_date','opening_drawdown','own_drawdown_change','decision_cash_weight','parent_risk_state','soxx_ret20','vix']].copy();firstbar['year']=firstbar.trade_date.dt.year;firstbar['quarter']=firstbar.trade_date.dt.to_period('Q').astype(str)
firstbar['candidate_soxx_score']=calc[first];firstbar['control_soxx_score']=base[first];firstbar['candidate_zero_soxx_score']=pred.candidate_zero_soxx.to_numpy(float)[first]
firstbar['actual_soxx_value']=labels_eval.soxx_value.to_numpy(float)[first]
firstbar['soxx_fee_fraction']=(labels_eval.soxx_fee_usd/labels_eval.decision_nav).to_numpy(float)[first]
firstbar['actual_gross_price_share_value']=firstbar.actual_soxx_value+firstbar.soxx_fee_fraction
firstbar['candidate_error']=firstbar.candidate_soxx_score-firstbar.actual_soxx_value
quarter=[]
for q,g in firstbar.groupby('quarter',sort=True):
 actions=daily.loc[daily.trade_date.dt.to_period('Q').astype(str).eq(q)]
 changed=(actions.arm_candidate.ne(actions.arm_control)|actions.bar_end_at_candidate.fillna(pd.Timestamp('1900-01-01')).ne(actions.bar_end_at_control.fillna(pd.Timestamp('1900-01-01')))|actions.raw_shares_candidate.ne(actions.raw_shares_control))
 quarter.append({'quarter':q,'days':len(g),'candidate_soxx_first_score_mean_bp':float(10000*g.candidate_soxx_score.mean()),'control_soxx_first_score_mean_bp':float(10000*g.control_soxx_score.mean()),'actual_soxx_first_value_mean_bp':float(10000*g.actual_soxx_value.mean()),'actual_soxx_first_fee_mean_bp':float(10000*g.soxx_fee_fraction.mean()),'actual_soxx_first_gross_price_share_mean_bp':float(10000*g.actual_gross_price_share_value.mean()),'candidate_overprediction_mean_bp':float(10000*g.candidate_error.mean()),'candidate_minus_same_model_zero_first_score_mean_bp':float(10000*(g.candidate_soxx_score-g.candidate_zero_soxx_score).mean()),'opening_drawdown_mean_pct':float(100*g.opening_drawdown.mean()),'own_drawdown_change_first_mean_pct_points':float(100*g.own_drawdown_change.mean()),'prior_soxx20_mean_pct':float(100*g.soxx_ret20.mean()),'vix_mean':float(g.vix.mean()),'candidate_first_0948_order_days':int((actions.bar_end_at_candidate.dt.hour.eq(9)&actions.bar_end_at_candidate.dt.minute.eq(48)).sum()),'candidate_control_changed_days':int(changed.sum()),'candidate_soxx_order_days':int(actions.arm_candidate.eq('SOXX').sum()),'candidate_local_value_sum':float(actions.value_candidate.sum()),'control_local_value_sum':float(actions.value_control.sum())})
quarter=pd.DataFrame(quarter).sort_values('quarter');assert quarter.quarter.tolist()==plan['all_quarters']
contrib=pd.DataFrame(z[first]*coef,columns=names);contrib['quarter']=firstbar.quarter.to_numpy()
a=contrib.loc[contrib.quarter.eq(plan['matched_stronger_period']),names].mean();b=contrib.loc[contrib.quarter.eq(plan['weak_period']),names].mean();shift=b-a
score_shift=float(firstbar.loc[firstbar.quarter.eq(plan['weak_period']),'candidate_soxx_score'].mean()-firstbar.loc[firstbar.quarter.eq(plan['matched_stronger_period']),'candidate_soxx_score'].mean())
max_shift_closure=abs(float(shift.sum())-score_shift);assert max_shift_closure<1e-12
feature_shift=pd.DataFrame({'feature':names,'contribution_weak_minus_stronger':shift.to_numpy(float)}).sort_values('contribution_weak_minus_stronger',key=np.abs,ascending=False)
feature_shift['family']=feature_shift.feature.map(lambda x:'original39' if x in source_plan['base39_features'] else 'state' if x in source_plan['state_features'] else 'ordinary_minute' if x in source_plan['ordinary_minute_features'] else 'own_drawdown_change' if x==source_plan['candidate_only_feature'] else 'missing_indicator')
family={str(k):float(v) for k,v in feature_shift.groupby('family').contribution_weak_minus_stronger.sum().items()}
quarter.to_csv(card/'quarter_firstbar_calibration.csv',index=False)
feature_shift.to_csv(card/'saved_model_feature_shift.csv',index=False)
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'source_recorder_id':plan['information_recorder_id'],'eval_rows':len(eval_panel),'eval_days':494,'firstbar_days':int(first.sum()),'saved_prediction_max_error':max_prediction_error,'manual_coefficient_prediction_max_error':max_manual_error,'weak_minus_stronger_score_shift':score_shift,'feature_shift_closure_error':max_shift_closure,'feature_family_shift':family,'top_abs_feature_shifts':feature_shift.head(10).to_dict('records'),'quarter_firstbar':quarter.to_dict('records'),'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'saved_model_calibration_and_first_action_failure_attributed_no_policy'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'parity':max_prediction_error,'shift_closure':max_shift_closure,'weak_quarter':quarter.loc[quarter.quarter.eq('2022Q2')].to_dict('records'),'stronger_quarter':quarter.loc[quarter.quarter.eq('2021Q3')].to_dict('records'),'feature_family_shift':family,'top_features':feature_shift.head(6).to_dict('records')},ensure_ascii=False,indent=2))

{
  "parity": 0.0,
  "shift_closure": 1.734723475976807e-18,
  "weak_quarter": [
    {
      "quarter": "2022Q2",
      "days": 62,
      "candidate_soxx_first_score_mean_bp": 91.72082537532025,
      "control_soxx_first_score_mean_bp": 89.44897646546272,
      "actual_soxx_first_value_mean_bp": -20.859123180257377,
      "actual_soxx_first_fee_mean_bp": 0.0203465459386877,
      "actual_soxx_first_gross_price_share_mean_bp": -20.838776634318684,
      "candidate_overprediction_mean_bp": 112.57994855557762,
      "candidate_minus_same_model_zero_first_score_mean_bp": -0.04939416664495952,
      "opening_drawdown_mean_pct": 23.51680655435041,
      "own_drawdown_change_first_mean_pct_points": -0.009758754100256825,
      "prior_soxx20_mean_pct": -5.501748733607406,
      "vix_mean": 27.222258229409494,
      "candidate_first_0948_order_days": 62,
      "candidate_control_changed_days": 0,
      "candidate_soxx_order_days": 62,
      "candidate_local_value_sum": -0.12932656371759574,
   

## 中文结论

SOXX历史Goal未通过。本卡零新拟合/账户，已保存2021/2022 Qlib候选与控制SOXX预测逐行最大误差0，Ridge截距＋标准化特征贡献最大误差0；全部8季度首分钟已知状态、预测、实际五会话费后值、费用及日动作见CSV/audit。2022Q2 62/62天两模型09:48即买SOXX，候选首分均值预测+91.721bp，真实−20.859bp，费用仅0.020bp，过预测112.580bp；当时已知SOXX前20日均−5.50%，模型仍给高正值。2021Q3预测+13.115bp、真实−0.268bp，候选仅7日不同动作但相对控制局部增益；2021Q4预测−23.582bp、真实+17.152bp，反向漏判。2021Q3→2022Q2模型分数净上移78.606bp，原39项+65.305bp、状态+13.694bp，普通分钟−0.455bp、新自身回撤仅+0.062bp；原39项内巨幅正负贡献互相抵消，不能解作因果。失败是旧输入映射跨年失准且分钟增量未穿过动作阈值，价格/持仓路径而非0.02bp费用造成Q2局部负值。事后季度/系数不得生成开关；固定版本仍关闭。下一步只读查“前收盘→当日已完成开盘”的信息是否已在旧路测试及能否点时可用，不能直接重训。

旧路核对追加：SOXL/SOXX隔夜开盘gap及gap×前15分钟收益已实际训练/完整账户失败，见closed_route_check.json。换成五会话标签不自动形成新信息；当前needs_distinct_decision_time_mechanism，不重拟合旧字段。